# Project Phase 2 Orchestrator Wrapper

Notebook wrapper for the `prototype` orchestrator so you can route to the 3 implemented stories from Jupyter.

In [ ]:
from prototype.orchestrator import AgenticOrchestrator

orchestrator = AgenticOrchestrator()
print("Orchestrator initialized.")

In [ ]:
def ask(user_query: str, thread_id: str = "default", show_story_output: bool = False, debug: bool = False):
    out = orchestrator.invoke(user_query, thread_id=thread_id)
    print("USER:", user_query)
    print("THREAD:", thread_id)
    print("\nASSISTANT:")
    print(out["response"])
    print(f"\n[routed domain={out['active_domain']} story={out['active_story_id']}]")

    if debug:
        rm = out.get("router_metrics", {})
        print("\nrouter_reason:", out.get("router_reason"))
        print("continuation_score:", rm.get("continuation_score"))
        print("domain_scores:", rm.get("domain_scores"))
        print("margin:", rm.get("margin"))

    if show_story_output and out.get("story_output") is not None:
        print("\nstory_output:")
        print(out["story_output"])

    return out


## Quick Tests

In [ ]:
_ = ask("Summarize last month feedback for CAMP_105 across app and email and suggest adjustments", thread_id="user_a")

In [ ]:
_ = ask("Am I improving in my workouts over the last 8 weeks for MB-001?", thread_id="user_a")

In [ ]:
_ = ask("Can you check my latest suspicious login alert for M001?", thread_id="user_b")

## Inspect State

In [ ]:
orchestrator.state

## Multi-Turn Conversation Test
Run this cell to simulate a conversation across multiple turns using the same orchestrator state.

In [ ]:
thread_id = "user_a"
test_turns = [
    "Hi, I want to understand my recent workouts for MB-001",
    "Also check if there were suspicious logins recently",
    "Now summarize last month feedback for CAMP_105 from app and email",
]

for i, turn in enumerate(test_turns, start=1):
    out = ask(turn, thread_id=thread_id)
    print(f"turn={i} | thread={thread_id} | domain={out['active_domain']} | story={out['active_story_id']}")
    print('-' * 80)

# Optional: inspect compact routing trace
orchestrator.state.route_trace[-6:]


## Interactive Multi-Turn Loop
Run this cell for live manual testing. Type `exit` or `quit` to stop.

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id)
    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)
